# AgentOps Lab 11 - Multi-agent implementation with AutoGen

AutoGen's AgentChat layer is useful for teaching multi-agent collaboration because it has explicit agents and team patterns. In particular, `SelectorGroupChat` dynamically chooses the next participant from shared context, which makes coordination visible.


## Notebook-first learning contract

This notebook is the primary lesson for this topic. The Python module is not a separate replacement for the lesson; it is the implementation layer that the notebook explains, runs, breaks, and evaluates. Work through the notebook in this order:

1. read the concept model and architecture boundary;
2. inspect the tool/state/policy contracts;
3. run the deterministic implementation;
4. trigger the deliberate failure case;
5. record evaluation, cost, latency, and safety observations; and
6. answer the architecture question before moving on.


## Deep-dive training guide — AutoGen selector-style teams

### Concepts to master

- shared team context
- dynamic next-speaker selection
- team loop budgets and ownership rules

### Implementation walkthrough

`autogen_selector_team.py` simulates a selector group chat and a failure loop, then documents the equivalent AutoGen AgentChat shape.

### Deliberate failure case

Remove ownership rules and let agents bounce responsibility: observability asks deployment, deployment asks observability, analyst asks both again.

### Learner exercise

Add a termination condition requiring both analyst recommendation and risk-review challenge before final answer.

### What to write down

For each run, capture the chosen architecture, tool trajectory, evidence used, rejected alternatives, stop condition, estimated cost, latency, and one sentence explaining whether the architecture was the least autonomous reliable option.


## Engineering checklist for this notebook

Use this checklist as your mini design review before you call the topic complete.

| Area | Question to answer |
| --- | --- |
| Control boundary | Which decisions are made by deterministic code, and which are delegated to the model? |
| Tools | Are tool inputs typed, narrow, authorized, and auditable? |
| State | What state is carried between steps, and what should never become long-term memory? |
| Failure mode | What is the easiest way this design loops, overacts, or fabricates certainty? |
| Evaluation | Which outcome, trajectory, safety, cost, and latency signals prove the design is working? |
| Architecture choice | Why is this architecture simpler or better than the nearest alternative? |


## Conceptual AutoGen implementation

```python
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat

observability_agent = AssistantAgent(
    "observability",
    description="Investigates metrics and logs",
    model_client=model,
)

deployment_agent = AssistantAgent(
    "deployment",
    description="Investigates deployment changes",
    model_client=model,
)

team = SelectorGroupChat(
    participants=[observability_agent, deployment_agent, customer_agent, analyst_agent],
    model_client=model,
    termination_condition=termination,
)

await team.run(task="Checkout conversion dropped 35% in Europe...")
```

The executable lab below is deterministic, so it runs without AutoGen or API credentials while preserving the coordination concepts.

```mermaid
sequenceDiagram
    participant S as Selector
    participant O as Observability
    participant D as Deployment
    participant C as Customer Impact
    participant A as Analyst
    participant R as Risk Reviewer
    S->>O: choose next speaker
    O-->>S: metrics and logs
    S->>D: choose next speaker
    D-->>S: release evidence
    S->>C: choose next speaker
    C-->>S: affected segments
    S->>A: synthesize
    A-->>S: likely cause and response
    S->>R: review risk
    R-->>A: challenge unsupported claims
```


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.autogen_selector_team import MAX_AGENT_TURNS, MAX_TEAM_MESSAGES, ownership_rules, run_failure_loop, run_selector_team


## Run the selector team

The selector chooses a participant based on the shared context. The run ends when the recommendation is ready.

In [ ]:
run = run_selector_team()
print(run.stopped_reason)
for message in run.messages:
    print(f"{message.speaker}: {message.content}")


## Explicit ownership

Selector teams need role boundaries. Without ownership, agents can bounce responsibility around instead of producing evidence.

In [ ]:
for rule in ownership_rules():
    print("-", rule)


## Deliberate failure loop

Now create a coordination failure:

- ObservabilityAgent: probably deployment
- DeploymentAgent: probably database
- Analyst: ask observability
- Observability: ask deployment

The fix is not a better vibe. Add budgets and ownership.

In [ ]:
failure = run_failure_loop()
print("stopped:", failure.stopped_reason)
print("messages:", len(failure.messages))
print("turns:", failure.turns_by_agent)
print("MAX_TEAM_MESSAGES:", MAX_TEAM_MESSAGES, "MAX_AGENT_TURNS:", MAX_AGENT_TURNS)


## Exercises

- Add a custom selector rule that sends the conversation to `risk_reviewer` whenever the analyst says "likely cause".
- Reduce `MAX_TEAM_MESSAGES` to 8. What useful evidence is lost?
- Add a candidate function that prevents `observability` and `deployment` from bouncing the task back and forth.
- Compare the deterministic selector run with a real AutoGen `SelectorGroupChat` implementation.

References: [AutoGen SelectorGroupChat](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/selector-group-chat.html), [AutoGen AgentChat agents](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/agents.html), and [AutoGen paper](https://arxiv.org/abs/2308.08155).